In [ ]:
# rende config.py (in notebooks/) importabile anche dai notebook in sottocartelle
import sys, os
_cfg = os.getcwd()
while _cfg != os.path.dirname(_cfg):
    if os.path.exists(os.path.join(_cfg, 'config.py')):
        sys.path.insert(0, _cfg)
        break
    _cfg = os.path.dirname(_cfg)
from config import CONFESS_DATA, BC_DATA, ERA5_ROOT, POST_DATA, WORK_DIR, FIG_DIR, FIG_DIR_2025
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.patches as mpatches

import concurrent.futures
from concurrent.futures import ProcessPoolExecutor

import logging

import albedo_functions as af

In [ ]:
# Configurazione logging
log_file = "05-Figx5_global_acc_seasons.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file, mode='w'),  # Write to file
    ]
)

In [ ]:
exp_ctrl = 'a1ua'
exp_sens = 'a52o'
var='tas'
era_var='2t'
COV_PATH = f'{POST_DATA}/'
LAI_PATH = f'{CONFESS_DATA}/'
SAVE_PATH = f'{WORK_DIR}/'
    
leads = [0, 1, 2, 3, 4]

In [ ]:
def main_script(season):
    logging.info(f"Starting main {season}")
    try:
        dset_ctrl = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_ctrl_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_ctrl_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        
        dset_ctrl_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_em = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_ctrl_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_em_land = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_land_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_ctrl_em_ocean= xr.open_mfdataset(f'{SAVE_PATH}/{exp_ctrl}_{var}_lead_*_1x1_global_acc_em_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        dset_sens_em_ocean = xr.open_mfdataset(f'{SAVE_PATH}/{exp_sens}_{var}_lead_*_1x1_global_acc_em_ocean_{season}_1999.nc', concat_dim='lead_time', combine='nested', chunks=-1)
        
        ld = xr.DataArray(leads, dims='lead_time', name='lead_time')
        dset_ctrl['lead_time']=ld
        dset_sens['lead_time']=ld
        dset_ctrl_em['lead_time']=ld
        dset_sens_em['lead_time']=ld
        
        dset_ctrl_land['lead_time']=ld
        dset_sens_land['lead_time']=ld
        dset_ctrl_em_land['lead_time']=ld
        dset_sens_em_land['lead_time']=ld
        
        
        dset_ctrl_ocean['lead_time']=ld
        dset_sens_ocean['lead_time']=ld
        dset_ctrl_em_ocean['lead_time']=ld
        dset_sens_em_ocean['lead_time']=ld
        
        # Convert the dataset to a DataFrame
        dset_ctrl = dset_ctrl.to_dataframe().reset_index()
        dset_sens = dset_sens.to_dataframe().reset_index()
        dset_ctrl_em = dset_ctrl_em.to_dataframe().reset_index()
        dset_sens_em = dset_sens_em.to_dataframe().reset_index()
        
        dset_ctrl_land = dset_ctrl_land.to_dataframe().reset_index()
        dset_sens_land = dset_sens_land.to_dataframe().reset_index()
        dset_ctrl_em_land = dset_ctrl_em_land.to_dataframe().reset_index()
        dset_sens_em_land = dset_sens_em_land.to_dataframe().reset_index()
        
        dset_ctrl_ocean = dset_ctrl_ocean.to_dataframe().reset_index()
        dset_sens_ocean = dset_sens_ocean.to_dataframe().reset_index()
        dset_ctrl_em_ocean = dset_ctrl_em_ocean.to_dataframe().reset_index()
        dset_sens_em_ocean = dset_sens_em_ocean.to_dataframe().reset_index()
        
        plot_figures(season, dset_ctrl, dset_ctrl_land, dset_ctrl_ocean, dset_sens, dset_sens_land, dset_sens_ocean, dset_ctrl_em, dset_ctrl_em_land, dset_ctrl_em_ocean, dset_sens_em, dset_sens_em_land, dset_sens_em_ocean)
        
        logging.info(f"Bias files written successfully for season {season}")

    except Exception as e:
        logging.exception(f"Error occurred for season {season}: {e}")

In [ ]:
def plot_figures(season, dset_ctrl, dset_ctrl_land, dset_ctrl_ocean, dset_sens, dset_sens_land, dset_sens_ocean, dset_ctrl_em, dset_ctrl_em_land, dset_ctrl_em_ocean, dset_sens_em, dset_sens_em_land, dset_sens_em_ocean):
    #global
    fig,ax=plt.subplots(figsize=(12, 6))
    sns.boxplot(x='lead_time', y='tas', data=dset_sens, ax=ax, color='lightcoral', linecolor='r')
    red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
    sns.boxplot(x='lead_time', y='tas', data=dset_ctrl, ax=ax, color='lavender', linecolor='b')
    blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
    plt.legend(handles=[red_patch, blue_patch],loc='upper right')
    
    dset_ctrl_em[var].plot(ax=ax,color='k', linestyle='--')
    dset_sens_em[var].plot(ax=ax,color='k')
    ax.set_title('a) GLOBAL')
    
    fig.savefig(f'{FIG_DIR}/{var}_acc_global_{season}_1999.png', dpi=600, bbox_inches = 'tight')

    #land    
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(x='lead_time', y='tas', data=dset_sens_land, ax=ax, color='lightcoral', linecolor='r')
    red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
    sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_land, ax=ax, color='lavender', linecolor='b')
    blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
    
    # Force the legend to the upper-right corner
    plt.legend(handles=[red_patch, blue_patch], loc='upper right', bbox_to_anchor=(1, 1))
    
    dset_ctrl_em_land[var].plot(ax=ax, color='k', linestyle='--')
    dset_sens_em_land[var].plot(ax=ax, color='k')
    ax.set_title('b) LAND ONLY')
    
    fig.savefig(f'{FIG_DIR}/{var}_acc_land_{season}_1999.png', dpi=600, bbox_inches='tight')

    #ocean
    fig,ax=plt.subplots(figsize=(12, 6))
    sns.boxplot(x='lead_time', y='tas', data=dset_sens_ocean, ax=ax, color='lightcoral', linecolor='r')
    red_patch = mpatches.Patch(color='lightcoral', label='DCP-SENS')
    sns.boxplot(x='lead_time', y='tas', data=dset_ctrl_ocean, ax=ax, color='lavender', linecolor='b')
    blue_patch = mpatches.Patch(color='lavender', label='DCP-CTRL')
    plt.legend(handles=[red_patch, blue_patch],loc='upper right')
    
    dset_ctrl_em_ocean[var].plot(ax=ax,color='k', linestyle='--')
    dset_sens_em_ocean[var].plot(ax=ax,color='k')
    ax.set_title('c) OCEAN ONLY')
    
    fig.savefig(f'{FIG_DIR}/{var}_acc_ocean_{season}_1999.png', dpi=600, bbox_inches = 'tight')

In [ ]:
%%time
seasons = ['DJF','MAM','JJA','SON']
futures = []

# Submit all (season, lead) combinations in parallel
with concurrent.futures.ProcessPoolExecutor() as executor:
    futures = [executor.submit(main_script, season) for season in seasons]
    # progresso: stampa ogni task man mano che termina
    for _i, _f in enumerate(concurrent.futures.as_completed(futures), 1):
        _f.result()
        print(f"  completato {_i}/{len(futures)}", flush=True)